# Physics Test Calculator

In [8]:
from dataclasses import dataclass
import numpy as np
from scipy import constants as const
import functools

In [9]:
@dataclass(frozen=True)
class Gas:
    name: str
    paschen_A: float     # cm^-1 Torr^-1
    paschen_B: float     # V cm^-1 Torr^-1
    ionization_ev: float
    
XENON = Gas("xenon", 26.0, 350.0, 12.13)

In [11]:
@dataclass
class Globe:
    """Noble gas sphere."""
    diameter_cm: float = 8.5     # outside diameter
    gas: Gas = XENON
    pressure_torr: float = 15.0
    wall_mm: float = 1.5         # glass wall thickness (estimate; ask the seller)
    ring_inset_mm: float = 6.0   # from the inner glass surface to the ring's center

    def __post_init__(self):
        if self.diameter_cm <= 0 or self.pressure_torr <= 0:
            raise ValueError("Diameter and pressure must be positive.")

    # --- Derived values, in SI ---
    @property
    def inner_radius(self) -> float:
        """Radius of the gas volume, in meters."""
        return self.diameter_cm / 200 - self.wall_mm / 1000

    @property
    def ring_radius(self) -> float:
        """Radius of the plasma ring, in meters."""
        return self.inner_radius - self.ring_inset_mm / 1000

    @property
    def gas_density(self) -> float:
        """Atoms per cubic meter, at 20 degrees C."""
        return self.pressure_torr * 133.322 / (const.k * 293.15)

In [12]:
@dataclass
class Coil:
    """Primary coil of the transformer model"""
    wire_diameter_thickness: float = 3.2
    turns: int = 2
    gap_mm: float = 3.0
    spacing_mm: float = 3.2
    
    def __post_init__(self):
        if self.turns < 1:
            raise ValueError("A coil needs at least one turn.")
        if self.spacing_mm <= 0 and self.turns > 1:
            raise ValueError("Turns can't touch: set spacing_mm above zero.")
    
    # --- Geometry, in SI ---
    @property
    def wire_radius(self) -> float:
        """Conductor radius a, in meters."""
        return self.wire_diameter_mm / 2000

    @property
    def pitch(self) -> float:
        """Center-to-center distance between adjacent turns, in meters."""
        return (self.wire_diameter_mm + self.spacing_mm) / 1000

    @property
    def length(self) -> float:
        """Axial length l, first turn's center to last turn's, in meters (for the formulas)."""
        return (self.turns - 1) * self.pitch

    @property
    def height_mm(self) -> float:
        """Outer height of the winding, in mm (for designing the collar)."""
        return self.turns * self.wire_diameter_mm + (self.turns - 1) * self.spacing_mm

    # --- Placement around a sphere ---
    def radius(self, globe: "Globe") -> float:
        """Coil radius R to the wire's center, in meters."""
        return globe.diameter_cm / 200 + self.gap_mm / 1000 + self.wire_radius

    def turn_positions(self, globe: "Globe") -> list[tuple[float, float]]:
        """(radius, height) of each turn's center, centered on the sphere's equator.
        Feed these to the mutual-inductance functions for the exact flux and L."""
        R = self.radius(globe)
        z0 = -self.length / 2
        return [(R, z0 + i * self.pitch) for i in range(self.turns)]
    

In [6]:
class Controller:
    """
    frequency
    """

## Finding Faraday's Law Values and Field Strengths

### Magnetic Field
The magnetic field through the center of the coil can be given as
$$B_{center} = \frac{\mu_0 N I}{\sqrt{\ell^2 + 4R^2}} \;\approx\; \frac{\mu_0 N I}{2R} \quad \text{when } \ell < 0.3\,D$$
For our coil, we will retain this assumption, as the height of the coil `ℓ` will likely be ***significantly*** shorter than the radius of the xenon globe `R`

### Magnetic Flux
The flux through the cross section defined by the coil is
$$\Phi \approx B_{center} \cdot \pi r^2$$

### Ring Voltage
With the flux we can find the voltage around the plasma ring with oscillation `ω` using
$$V_{loop} = \omega \Phi$$

### Field Along the Ring
The electric field along the circumference given the ring voltage is given by
$$E = \frac{V_{loop}}{2\pi r} \;\approx\; \frac{\omega\, \mu_0 N I\, r}{4R}$$


In [ ]:
class EM:
    """Electromagnetism equations"""
    # Magnetism Equations
    @staticmethod
    @functools.lru_cache(maxsize=16)
    def b_field(turns, current, radius) -> float:
        """Magnetic field at the center of a flat coil, in tesla."""
        return const.mu_0 * turns * current / (2 * radius)

    @staticmethod
    @functools.lru_cache(maxsize=16)
    def b_flux(field, sphere_radius) -> float:
        """Magnetic flux from the coil going through the sphere, in webers"""
        return field * const.pi * sphere_radius ** 2

    # Electricity Equations
    @staticmethod
    @functools.lru_cache(maxsize=16)
    def v_loop(flux, w) -> float:
        """Voltage across coil loop with specified oscillation and field flux"""
        return flux * w
    
    @staticmethod
    @functools.lru_cache(maxsize=16)
    def e_field(loop_voltage, radius) -> float:
        """Field along circumference of circle with specified radius"""
        return loop_voltage / (2 * const.pi * radius)
    

In [ ]:
class Ionizer:
    gas: Gas = XENON
    
    @functools.cache
    @property
    @classmethod
    def A(cls, a_const):
        cls._A = a_const